# Jeux répétés : le tournoi et la fragilité de la coopération

[← 14 — Jeux différentiels](GameTheory-14-DifferentialGames-Python.ipynb) · [↑ Série GameTheory](README.md) · [15 — Jeux coopératifs →](GameTheory-15-CooperativeGames-Python.ipynb)


Le dilemme du prisonnier **épisode unique** a un équilibre strict : la défection commune. Le palier [04](GameTheory-04-MixedNash-Python.ipynb) l'a calculé, le palier [02](GameTheory-02-NormalForm-Python.ipynb) l'a éliminé par stratégies dominées. Ce carnet répond à la question naturelle qui suit : **que se passe-t-il quand le jeu se répète ?**

Trois résultats, tous mesurés dans ce carnet :

1. **Horizon fini** : la récursion arrière dénoue la coopération — si le nombre de tours est connu, l'équilibre parfait en sous-jeux est la défection dès le premier coup.
2. **Horizon infini (escompté)** : la coopération devient soutenable — on calcule le seuil exact du facteur d'escompte au-dessus duquel la stratégie *rancunier* rend la défection non profitable.
3. **Tournoi** : huit stratégies s'affrontent en round-robin, puis le même tournoi est rejoué avec du **bruit d'exécution** — la stratégie gagnante sans bruit s'effondre quand les coups sont mal transmis, et l'effondrement est mesuré, pas raconté.

Tout ce carnet est **auto-contenu** : moteur de match, stratégies et mesures sont écrits ici, en Python pur (`numpy` pour le seul tirage aléatoire). Chaque sortie est celle de notre simulation ; aucune valeur n'est recopiée d'ailleurs.


Le dilemme du prisonnier **épisode unique** a un équilibre strict : la défection commune. Le palier [02](GameTheory-02-NormalForm-Python.ipynb) l'a éliminée par stratégies dominées, le palier [04](GameTheory-04-NashEquilibrium-Python.ipynb) l'a calculée. Ce carnet répond à la question naturelle qui suit : **que se passe-t-il quand le jeu se répète ?**

**Position dans la série.** Le sujet existe déjà sous deux angles : [06](GameTheory-06-EvolutionTrust-Python.ipynb) le traite de façon **évolutionnaire** (tournois, dynamique du réplicateur, librairie Axelrod) et [06c](GameTheory-06c-RepeatedGames-FolkTheorem-Python.ipynb) démontre le théorème folk, dont le seuil critique est prouvé en Lean au palier [06b](GameTheory-06b-Lean-RepeatedGames-Lean-Python.ipynb). Ce carnet isole le **mécanisme d'équilibre** et le met au banc, en Python pur et sans dépendance : pourquoi l'horizon fini dénoue la coopération, quel seuil de patience exact la rétablit, et ce que le bruit fait au classement — chaque affirmation mesurée dans les sorties :

1. **Horizon fini** : la récursion arrière dénoue la coopération — si le nombre de tours est connu, l'équilibre parfait en sous-jeux est la défection dès le premier coup.
2. **Horizon infini (escompté)** : la coopération devient soutenable — on calcule le seuil exact du facteur d'escompte au-dessus duquel la stratégie *rancunier* rend la défection non profitable, et on le vérifie numériquement de part et d'autre.
3. **Tournoi** : huit stratégies s'affrontent en round-robin, puis le même tournoi est rejoué avec du **bruit d'exécution** multi-graine — la stratégie gagnante sans bruit s'effondre quand les coups sont mal transmis, et l'effondrement est mesuré, pas raconté.

Tout ce carnet est **auto-contenu** : moteur de match, stratégies et mesures sont écrits ici, en Python pur (`numpy` pour le seul tirage aléatoire). Chaque sortie est celle de notre simulation ; aucune valeur n'est recopiée d'ailleurs.

In [1]:
T, R, P, S = 5, 3, 1, 0  # tentation, recompense, punition, dupe
GAINS = {('C', 'C'): (R, R), ('C', 'D'): (S, T), ('D', 'C'): (T, S), ('D', 'D'): (P, P)}

print('Matrice (gain du joueur ligne, gain du joueur colonne) :')
print('            colonne C      colonne D')
print(f'ligne C   {GAINS[("C","C")]}      {GAINS[("C","D")]}')
print(f'ligne D   {GAINS[("D","C")]}      {GAINS[("D","D")]}')

# (D, D) est-il un equilibre en strategies strictement dominantes ?
# Pour chaque joueur, on compare C et D contre CHAQUE action de l'autre.
for opposant in ('C', 'D'):
    gain_c = GAINS[('C', opposant)][0]
    gain_d = GAINS[('D', opposant)][0]
    verdict = 'D strictement meilleur' if gain_d > gain_c else 'pas de domination stricte'
    print(f'contre un adversaire {opposant} : C rapporte {gain_c}, D rapporte {gain_d} -> {verdict}')

Matrice (gain du joueur ligne, gain du joueur colonne) :
            colonne C      colonne D
ligne C   (3, 3)      (0, 5)
ligne D   (5, 0)      (1, 1)
contre un adversaire C : C rapporte 3, D rapporte 5 -> D strictement meilleur
contre un adversaire D : C rapporte 0, D rapporte 1 -> D strictement meilleur


La défection **domine strictement** la coopération : contre un adversaire qui coopère, D rapporte $T=5 > R=3$ ; contre un adversaire qui fait défection, D rapporte $P=1 > S=0$. L'équilibre de l'épisode unique est donc $(D, D)$ avec gain $1$ chacun — alors que $(C, C)$ rapporterait $3$ chacun. C'est exactement le paradoxe : l'équilibre est dominé, au sens commun, par un profil non credible en stratégies dominantes. La répétition est le mécanisme qui rend la coopération **credible**.

## 2. Horizon fini : le dénouement par récursion arrière

Supposons le jeu répété $T$ fois, $T$ connu des deux joueurs. On calcule l'équilibre parfait en sous-jeux par induction **depuis le dernier tour**. À chaque tour restant, on demande à chaque joueur sa meilleure réponse, la valeur de continuation étant déjà connue — et comme les gains des tours restants ne dépendent pas de l'action courante autrement que par cette valeur, la meilleure réponse au tour $k$ est celle de l'épisode unique.

In [2]:
def meilleure_reponse1(suite, action_adversaire):
    # Meilleure reponse du joueur 1 : sa decision est la PREMIERE composante de la cle.
    if suite[('C', action_adversaire)] > suite[('D', action_adversaire)]:
        return 'C'
    return 'D'

def meilleure_reponse2(suite, action_adversaire):
    # Meilleure reponse du joueur 2 : sa decision est la SECONDE composante de la cle.
    if suite[(action_adversaire, 'C')] > suite[(action_adversaire, 'D')]:
        return 'C'
    return 'D'

def equilibre_tour(valeur_suite):
    # Profil du tour courant : meilleures reponses mutuelles, la suite etant fixee.
    suite1 = {(x, y): GAINS[(x, y)][0] + valeur_suite[(x, y)] for x in 'CD' for y in 'CD'}
    suite2 = {(x, y): GAINS[(x, y)][1] + valeur_suite[(x, y)] for x in 'CD' for y in 'CD'}
    for a1 in 'CD':
        for a2 in 'CD':
            if meilleure_reponse1(suite1, a2) == a1 and meilleure_reponse2(suite2, a1) == a2:
                return (a1, a2)

for horizon in (1, 2, 3, 5, 10):
    # valeur de la suite apres le dernier tour : sous-jeu vide, gain nul
    valeur_suite = {(a1, a2): 0.0 for a1 in 'CD' for a2 in 'CD'}
    profil_premier = None
    for _ in range(horizon):  # du dernier tour vers le premier
        eq = equilibre_tour(valeur_suite)
        profil_premier = eq  # le dernier profil calcule est celui du tour 1
        # les sous-jeux de l'etape suivante ont la meme valeur d'equilibre :
        # valeur du sous-jeu restant = gain d'equilibre du tour + suite deja connue
        valeur_suite = {(x, y): GAINS[eq][0] + valeur_suite[(x, y)] for x in 'CD' for y in 'CD'}
    print(f'horizon {horizon:>2} : profil au tour 1 = {profil_premier}, '
          f'valeur totale par joueur = {valeur_suite[profil_premier]:.1f}')

horizon  1 : profil au tour 1 = ('D', 'D'), valeur totale par joueur = 1.0
horizon  2 : profil au tour 1 = ('D', 'D'), valeur totale par joueur = 2.0
horizon  3 : profil au tour 1 = ('D', 'D'), valeur totale par joueur = 3.0
horizon  5 : profil au tour 1 = ('D', 'D'), valeur totale par joueur = 5.0
horizon 10 : profil au tour 1 = ('D', 'D'), valeur totale par joueur = 10.0


Quel que soit l'horizon fini testé — 1, 2, 3, 5 ou 10 — le profil d'équilibre parfait au **premier** tour est $(D, D)$. C'est le dénouement (*unraveling*) : au dernier tour, aucune menace n'a de valeur (il n'y a plus de suite), donc chacun fait défection ; sachant cela, l'avant-dernier tour n'a plus rien à protéger, donc défection ; et ainsi de suite jusqu'au premier coup. **Connaître la fin du jeu détruit la coopération.** La sortie le montre sur cinq horizons : la valeur totale croît linéairement avec l'horizon (l'horizon multiplié par le gain de punition $P = 1$ : 1, 2, 3, 5, 10), et le gain de coopération $R$ n'apparaît jamais.

## 3. Horizon infini : le seuil d'escompte qui soutient la coopération

Si le jeu se répète sans fin connue, avec un facteur d'escompte $\delta \in [0,1)$ (un gain dans $k$ tours vaut $\delta^k$ fois son montant), la menace redevient crédible. Considérons la stratégie **rancunier** (*grim trigger*) : coopérer tant que personne n'a fait défection, sinon défection pour toujours.

Sur le chemin de coopération, dévier une fois rapporte $T$ puis $P$ à jamais ; rester coopératif rapporte $R$ à jamais. La déviation est non profitable si :

$$T + \sum_{k \geq 1} \delta^k P \;\leq\; \sum_{k \geq 0} \delta^k R
\quad\Longleftrightarrow\quad
\delta \;\geq\; \frac{T - R}{T - P} = \delta^\star$$

Le calcul ci-dessous mesure $\delta^\star$ pour notre matrice, puis **vérifie le seuil numériquement** en comparant les valeurs escomptées tronquées à 500 tours, de part et d'autre du seuil.

In [3]:
delta_star = (T - R) / (T - P)
print(f'seuil theorique  delta* = (T - R)/(T - P) = ({T}-{R})/({T}-{P}) = {delta_star:.4f}')

HORIZON = 500  # troncature : les termes au-dela pesent delta^500 < 1e-175 pour delta <= 0.99

def valeur_escomptee(gains_par_tour, delta, horizon=HORIZON):
    # Somme des gains par tour, escomptee, tronquee a `horizon` tours.
    return sum(delta ** k * g for k, g in enumerate(gains_par_tour[:horizon]))

print(f'\n{"delta":>6} {"cooperee":>10} {"deviation":>10} {"verdict":>28}')
for delta in (0.40, 0.49, 0.50, 0.51, 0.60):
    coop = valeur_escomptee([R] * HORIZON, delta)                      # rester sur le chemin
    devi = valeur_escomptee([T] + [P] * (HORIZON - 1), delta)          # devier une fois puis puni
    ecart = devi - coop  # > 0 : la deviation rapporte davantage
    if abs(ecart) < 1e-9:
        verdict = 'indifference au seuil'
    elif ecart > 0:
        verdict = 'deviation profitable'
    else:
        verdict = 'cooperation soutenable'
    print(f'{delta:>6.2f} {coop:>10.3f} {devi:>10.3f} {verdict:>28}')

seuil theorique  delta* = (T - R)/(T - P) = (5-3)/(5-1) = 0.5000

 delta   cooperee  deviation                      verdict
  0.40      5.000      5.667         deviation profitable
  0.49      5.882      5.961         deviation profitable
  0.50      6.000      6.000        indifference au seuil
  0.51      6.122      6.041       cooperation soutenable
  0.60      7.500      6.500       cooperation soutenable


Le seuil théorique est $\delta^\star = 0{,}5$ et la vérification numérique s'y colle exactement : à $\delta = 0{,}40$ et $0{,}49$ la déviation rapporte davantage (l'écart se referme quand $\delta$ monte), à $\delta = 0{,}51$ et $\delta = 0{,}60$ la coopération devient la meilleure réponse, et à $\delta = 0{,}50$ l'écart passe sous $10^{-9}$ — le verdict rendu est *indifférence au seuil*, ce qui est exactement la définition de $\delta^\star$ : le point où dévier ne rapporte plus rien mais ne coûte rien encore. C'est la version quantitative du **théorème folk** pour notre matrice : n'importe quel gain faisable individuellement rationnel — y compris $R$ — devient un équilibre pour un joueur assez patient (la démonstration formelle vit en Lean au palier [06b](GameTheory-06b-Lean-RepeatedGames-Lean-Python.ipynb), l'argument géométrique au palier [06c](GameTheory-06c-RepeatedGames-FolkTheorem-Python.ipynb)). La coopération ne demande pas de bonté, seulement de la patience : les deux joueurs doivent accorder suffisamment de poids au futur.

## 4. Le moteur du tournoi

Une stratégie est une fonction qui, à chaque tour, regarde les deux historiques (le sien et celui de l'adversaire) et rend 'C' ou 'D'. Le moteur joue un match en N tours, avec un **bruit d'exécution** optionnel : avec probabilité p, l'action effectivement jouée est l'inverse de celle choisie — c'est le bruit le plus défavorable à la réciprocité, car une erreur de transmission est vue par l'adversaire comme une trahison.

In [4]:
import numpy as np

def jouer_match(strat1, strat2, n_tours, bruit=0.0, rng=None):
    # Joue un match ; rend (score1, score2). bruit : proba d'inverser l'action jouee.
    h1, h2 = [], []  # historiques des actions EFFECTIVEMENT jouees
    s1 = s2 = 0
    for _ in range(n_tours):
        a1, a2 = strat1(h1, h2), strat2(h2, h1)
        if bruit > 0 and rng is not None:
            if rng.random() < bruit:
                a1 = 'D' if a1 == 'C' else 'C'
            if rng.random() < bruit:
                a2 = 'D' if a2 == 'C' else 'C'
        g1, g2 = GAINS[(a1, a2)]
        s1 += g1
        s2 += g2
        h1.append(a1)
        h2.append(a2)
    return s1, s2

def toujours_c(moi, autre):  return 'C'
def toujours_d(moi, autre):  return 'D'
def aleatoire(moi, autre):   return 'C' if rng_global.random() < 0.5 else 'D'
def tft(moi, autre):         return autre[-1] if autre else 'C'
def gtft(moi, autre):
    if not autre or autre[-1] == 'C':
        return 'C'
    return 'C' if rng_global.random() < 1/3 else 'D'  # indulgence : pardonner un tiers des defauts
def rancunier(moi, autre):  return 'D' if 'D' in autre else 'C'
def pavlov(moi, autre):
    if not autre:
        return 'C'
    dernier_gain = GAINS[(moi[-1], autre[-1])][0]
    return moi[-1] if dernier_gain in (R, T) else ('D' if moi[-1] == 'C' else 'C')  # rester si gain, changer si perte
def sondeur(moi, autre):
    if len(moi) < 3:
        return ['D', 'C', 'C'][len(moi)]
    if autre[1:3] == ['C', 'C']:  # l'adversaire n'a pas puni la sonde
        return 'D'
    return autre[-1]

rng_global = np.random.default_rng(0)
STRATEGIES = {
    'ToujoursC': toujours_c, 'ToujoursD': toujours_d, 'Aleatoire': aleatoire,
    'TitForTat': tft, 'GenereuxTFT': gtft, 'Rancunier': rancunier,
    'Pavlov': pavlov, 'Sondeur': sondeur,
}

# essai du moteur : TitForTat contre ToujoursD, 5 tours, sans bruit
s_tft, s_tdd = jouer_match(tft, toujours_d, 5)
print(f'moteur OK : TitForTat {s_tft}, ToujoursD {s_tdd}')
print('(attendu : TitForTat -- dupe (0) au tour 1 puis punition mutuelle (1 x 4) = 4 ;')
print('           ToujoursD -- tentation (5) au tour 1 puis punition mutuelle (1 x 4) = 9)')

moteur OK : TitForTat 4, ToujoursD 9
(attendu : TitForTat -- dupe (0) au tour 1 puis punition mutuelle (1 x 4) = 4 ;
           ToujoursD -- tentation (5) au tour 1 puis punition mutuelle (1 x 4) = 9)


Le contrôle du moteur donne exactement les valeurs attendues : Tit-for-Tat, qui commence par coopérer puis copie, subit la dupe ($S=0$) au premier tour puis se résigne à la punition mutuelle ($P=1$ pendant quatre tours), pour un total de $0 + 4 = 4$ ; Toujours-D récolte la tentation puis la même punition, $5 + 4 \times 1 = 9$. Chaque match bilatéral a donc un vainqueur clair — mais le tournoi ne classe pas les vainqueurs de matchs : il classe les totaux. Le moteur est prêt.

## 5. Le tournoi sans bruit

Round-robin complet : chaque paire de stratégies se rencontre dans les **deux ordres** (les stratégies à mémoire d'état comme Sondeur ne sont pas symétriques), $N=200$ tours par match, graine fixée. Le classement se lit en **score moyen par tour** — une métrique indépendante du nombre de tours.

In [5]:
N_TOURS = 200

def tournoi(bruit=0.0, graine=0, n_tours=N_TOURS):
    """Round-robin dans les deux ordres ; rend {nom: score moyen par tour}."""
    global rng_global
    rng_global = np.random.default_rng(graine)
    scores = {nom: 0.0 for nom in STRATEGIES}
    noms = list(STRATEGIES)
    for i, n1 in enumerate(noms):
        for n2 in noms[i:]:
            if n1 == n2:
                s, _ = jouer_match(STRATEGIES[n1], STRATEGIES[n2], n_tours, bruit, rng_global)
                scores[n1] += 2 * s
            else:
                sA, sB = jouer_match(STRATEGIES[n1], STRATEGIES[n2], n_tours, bruit, rng_global)
                sB2, sA2 = jouer_match(STRATEGIES[n2], STRATEGIES[n1], n_tours, bruit, rng_global)
                scores[n1] += sA + sA2
                scores[n2] += sB + sB2
    nb_matchs = 2 * len(STRATEGIES)  # chaque adversaire rencontre deux fois
    return {nom: scores[nom] / (nb_matchs * n_tours) for nom in scores}

resultats_sans_bruit = tournoi(bruit=0.0, graine=0)
classement = sorted(resultats_sans_bruit.items(), key=lambda kv: -kv[1])
print(f'{"rang":>4} {"strategie":<12} {"score/tour":>10}')
for rang, (nom, score) in enumerate(classement, 1):
    print(f'{rang:>4} {nom:<12} {score:>10.4f}')

rang strategie    score/tour
   1 TitForTat        2.6722
   2 GenereuxTFT      2.6003
   3 Rancunier        2.4956
   4 Pavlov           2.4747
   5 Sondeur          2.3363
   6 ToujoursD        2.1662
   7 Aleatoire        2.0641
   8 ToujoursC        2.0503


Sans bruit, le podium est structuré par la **réciprocité** : les stratégies qui coopèrent entre elles et punissent précisément (Tit-for-Tat, Généreux-TFT, Rancunier, Pavlov) trustent le haut du classement, parce qu'elles récoltent $R$ entre elles et ne se laissent pas exploiter longtemps par les prédateurs. Toujours-D, qui domine chaque match **bilateral** contre un donneur, paie cette rapacité dans le classement : contre ses pairs elle ne récolte que $P$, et contre les stratégies punitives elle perd la prime de tentation après un seul tour. C'est la leçon du tournoi : on gagne un round-robin en **maximisant le total**, pas en gagnant chaque match — coopérer avec ses semblables rapporte plus qu'exploiter les faibles.

## 6. Le bruit d'exécution : la fragilité mesurée

Rejouons le tournoi avec un bruit $p \in \{0{,}02, 0{,}05, 0{,}10\}$ sur l'action effectivement jouée. Pour Tit-for-Tat, ce bruit est le pire scénario : une coopération transmise comme défection déclenche un **cycle d'échanges de punitions** — deux Tit-for-Tat peuvent s'échanger des défections alternées indéfiniment après une seule erreur. Quatre graines ($0, 1, 7, 42$) par niveau de bruit : le classement est une moyenne, l'écart-type entre graines est affiché.

In [6]:
import statistics

niveaux_bruit = [0.0, 0.02, 0.05, 0.10]
graines = [0, 1, 7, 42]

# score moyen multi-graine par (strategie, niveau de bruit)
table = {nom: {} for nom in STRATEGIES}
for p in niveaux_bruit:
    for nom in STRATEGIES:
        essais = [tournoi(bruit=p, graine=g)[nom] for g in graines]
        table[nom][p] = (statistics.mean(essais), statistics.pstdev(essais))

print('score/tour moyen (ecart-type entre graines entre parentheses)\n')
entete = 'strategie'.ljust(12) + ''.join(f'bruit p={p:<5}'.ljust(20) for p in niveaux_bruit)
print(entete)
for nom in sorted(STRATEGIES, key=lambda n: -table[n][0.0][0]):
    ligne = nom.ljust(12)
    for p in niveaux_bruit:
        m, e = table[nom][p]
        ligne += f'{m:.3f} (+-{e:.3f})'.ljust(20)
    print(ligne)

print('\nRang de TitForTat par niveau de bruit :')
for p in niveaux_bruit:
    classement_p = sorted(STRATEGIES, key=lambda n: -table[n][p][0])
    print(f'  p = {p:.2f} : rang {classement_p.index("TitForTat") + 1} '
          f'(premier : {classement_p[0]}, {table[classement_p[0]][p][0]:.3f})')

score/tour moyen (ecart-type entre graines entre parentheses)

strategie   bruit p=0.0         bruit p=0.02        bruit p=0.05        bruit p=0.1         
TitForTat   2.660 (+-0.007)     2.210 (+-0.025)     2.156 (+-0.057)     2.144 (+-0.047)     
GenereuxTFT 2.516 (+-0.081)     2.229 (+-0.030)     2.134 (+-0.063)     2.015 (+-0.052)     
Rancunier   2.502 (+-0.004)     2.229 (+-0.059)     2.240 (+-0.017)     2.326 (+-0.017)     
Pavlov      2.472 (+-0.002)     2.213 (+-0.046)     2.112 (+-0.065)     2.122 (+-0.089)     
Sondeur     2.284 (+-0.042)     2.284 (+-0.061)     2.286 (+-0.090)     2.216 (+-0.066)     
ToujoursD   2.177 (+-0.015)     2.202 (+-0.015)     2.252 (+-0.031)     2.291 (+-0.025)     
Aleatoire   2.063 (+-0.033)     1.990 (+-0.046)     2.072 (+-0.045)     2.053 (+-0.062)     
ToujoursC   2.063 (+-0.010)     1.661 (+-0.094)     1.616 (+-0.037)     1.720 (+-0.066)     

Rang de TitForTat par niveau de bruit :
  p = 0.00 : rang 1 (premier : TitForTat, 2.660)
  p = 0.02

La fragilité annoncée est dans le tableau, et elle est **chiffrée** : premier du tournoi sans bruit, Tit-for-Tat tombe au **rang 5 dès $p = 0{,}02$**. Son score décroît de 2,660 à 2,144 entre $p = 0$ et $p = 0{,}10$ (environ $-0{,}52$ par tour). La raison se lit dans la mécanique de la section 4 : Tit-for-Tat n'a aucune mémoire au-delà du dernier coup, donc une erreur crée une rancune que seule une autre erreur peut retourner — les deux joueurs paient un duel de punitions tour après tour.

Le tableau montre un second mouvement, moins attendu : sous bruit fort, ce sont les stratégies **peu conciliantes** qui résistent le mieux. Rancunier ne perd que $0{,}18$ sur la même plage et prend la première place à $p = 0{,}10$ ; Toujours-D voit son score **monter** (2,177 à 2,291) — le bruit lui offre gratuitement les duels de punitions adverses ; Sondeur, insensible à la rancune par construction, tient la tête à $p = 0{,}02$ et $p = 0{,}05$. Généreux-TFT fait à peine mieux que Tit-for-Tat à faible bruit (2,229 contre 2,210 à $p = 0{,}02$) : pardonner casse certains cycles mais expose aux prédateurs — l'indulgence atténue la fragilité, elle ne l'annule pas.

La leçon dépasse le jeu : **une stratégie réciproque stricte, gagnante en environnement propre, s'effondre dès que le canal devient bruité** — et la robustesse ne s'achète pas qu'en indulgence : elle se gagne aussi en choisissant ce qu'on refuse de punir.

## 7. Exercices

### Exercice 1 — La stratégie majoritaire

`MajoriteC` coopère tant que l'adversaire a coopéré **au moins** la moitié du temps (strictement plus de la moitié de ses coups sont des C), et fait défection sinon. Implémente-la dans la fonction ci-dessous, puis soumet-la au tournoi : doit-elle être classée comme une stratégie *dure* (comme Rancunier) ou *indulgente* (comme Généreux-TFT) ?

In [7]:
def majorite_c(moi, autre):
    # TODO : rendre 'C' si l'adversaire a coopere strictement plus d'une fois sur deux, 'D' sinon
    # Indice : autre.count('C') et len(autre) suffisent ; que faire a la premiere equalite (tour 1) ?
    print('Exercice a completer : implanter MajoriteC puis la soumettre au tournoi')
    return None

# Etape 1 : verifier la strategie sur des historiques simples
# Etape 2 : l'ajouter au dict STRATEGIES sous le nom 'MajoriteC'
# Etape 3 : relancer tournoi() et comparer son rang a Rancunier et GenereuxTFT
print('historique C,C,D -> attendu C ;', 'obtenu :', majorite_c([], ['C', 'C', 'D']))

Exercice a completer : implanter MajoriteC puis la soumettre au tournoi
historique C,C,D -> attendu C ; obtenu : None


### Exercice 2 — Le seuil de bruit où Tit-for-Tat perd sa place

Le tableau de la section 6 montre quatre niveaux de bruit. Entre $p=0$ et $p=0{,}10$, il existe un seuil $p^\star$ où Tit-for-Tat quitte le podium. Balaye $p$ par pas de $0{,}01$ et retourne le premier $p$ où son rang dépasse 3.

In [8]:
def seuil_perte_podium(pas=0.01, graines_exo=(0, 1, 7, 42)):
    # TODO : pour p de 0 a 0.15 par pas de `pas`, calculer le rang moyen de TitForTat multi-graine
    # Indice : reutiliser tournoi(bruit=p, graine=g) et le classement comme en section 6
    print('Exercice a completer : balayer p et retourner le premier rang > 3')
    return None

print('seuil p* :', seuil_perte_podium())

Exercice a completer : balayer p et retourner le premier rang > 3
seuil p* : None


### Exercice 3 — Le seuil de patience sur une autre matrice

Reprends la question de la section 3 avec la matrice $T=4$, $R=2$, $P=1$, $S=0$ (une tentation plus faible). Calcule le $\delta^\star$ théorique, puis vérifie-le numériquement comme en section 3 : le seuil monte-t-il ou descend-il, et pourquoi ?

In [9]:
def delta_star_autre_matrice():
    # TODO : calculer (T2 - R2)/(T2 - P2) pour T2, R2, P2 = 4, 2, 1
    # Indice : la tentation relative (T-R) diminue alors que la punition relative (T-P) aussi ;
    # Etape 2 : verifier avec valeur_escomptee() comme en section 3, de part et d'autre du seuil
    print('Exercice a completer : calculer et verifier le seuil pour (T, R, P, S) = (4, 2, 1, 0)')
    return None

print('delta* attendu pour (4, 2, 1, 0) :', delta_star_autre_matrice())

Exercice a completer : calculer et verifier le seuil pour (T, R, P, S) = (4, 2, 1, 0)
delta* attendu pour (4, 2, 1, 0) : None


## Synthèse

Trois mécanismes séparent le dilemme répété de l'épisode unique, et chacun a été **mesuré** ici :

- **Horizon fini** : la récursion arrière dénoue la coopération tour après tour — cinq horizons testés, l'équilibre au tour 1 est toujours $(D, D)$.
- **Horizon infini** : la patience rend la menace crédible — seuil exact $\delta^\star = (T-R)/(T-P) = 0{,}5$ pour notre matrice, vérifié de part et d'autre par valeurs escomptées tronquées, avec l'indifférence exacte au seuil.
- **Tournoi** : la réciprocité gagne en environnement propre et s'effondre en environnement bruité — Tit-for-Tat passe de premier à cinquième dès $p = 0{,}02$, pendant que Rancunier et Toujours-D résistent.

**Ce qu'on emporte** : la coopération n'est pas un trait de caractère mais une propriété de la structure du jeu — longueur d'horizon, patience, fiabilité du canal. Changer un de ces trois paramètres suffit à faire basculer l'équilibre, dans un sens ou dans l'autre.

**Note bibliographique** : ce carnet est auto-contenu et n'attribue aucun résultat historique à une publication — toutes les valeurs affichées proviennent de la simulation locale. L'ancrage historique des tournois de jeux répétés (Axelrod, 1980-1984) attend le dépôt de la copie de référence au gisement bibliographique partagé ; il sera ajouté à cette section au moment du dépôt.